# Tutorial 5 — RDKit from Scratch
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

RDKit is the Swiss Army knife of cheminformatics. This tutorial covers everything you need to get started: loading molecules, computing descriptors, drawing structures, and writing your first pipeline.

In [ ]:
!pip install rdkit pandas numpy matplotlib pillow -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, rdMolDescriptors, AllChem
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, Image
import pandas as pd
import numpy as np

# Load from SMILES
mol = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")  # Aspirin
print(f"Aspirin: {mol.GetNumAtoms()} atoms, {mol.GetNumBonds()} bonds")
print(f"Formula: C{sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==6)}")

# Sanitize and get canonical SMILES
canonical = Chem.MolToSmiles(mol)
print(f"Canonical SMILES: {canonical}")

# Load from InChI
from rdkit.Chem.inchi import MolFromInchi
mol2 = MolFromInchi("InChI=1S/C9H8O4/c1-6(10)13-8-5-3-2-4-7(8)9(11)12/h2-5H,1H3,(H,11,12)")
print(f"From InChI: {Chem.MolToSmiles(mol2)}")

## Compute descriptors for a library

In [ ]:
library = {
    "Aspirin":       "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen":     "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Paracetamol":   "CC(=O)Nc1ccc(O)cc1",
    "Caffeine":      "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "Dopamine":      "NCCc1ccc(O)c(O)c1",
    "Testosterone":  "CC12CCC3C(C1CCC2=O)CCC4=CC(=O)CCC34C",
    "Chloroquine":   "CCN(CC)CCCC(C)Nc1ccnc2cc(Cl)ccc12",
    "Remdesivir":    "CCC(CC)COC(=O)c1ccc(N)nc1NC(=O)[C@@H](C)OP(=O)(OC[C@H]2O[C@@](C#N)(c3ccc4c(N)ncnc4n3)[C@@H](O)[C@@H]2O)Oc1ccccc1",
}

rows = []
for name, smi in library.items():
    mol = Chem.MolFromSmiles(smi)
    if mol is None: continue
    rows.append({
        "Name":     name,
        "MW":       round(Descriptors.ExactMolWt(mol), 2),
        "LogP":     round(Descriptors.MolLogP(mol), 2),
        "TPSA":     round(Descriptors.TPSA(mol), 1),
        "HBD":      rdMolDescriptors.CalcNumHBD(mol),
        "HBA":      rdMolDescriptors.CalcNumHBA(mol),
        "RotBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "Rings":    rdMolDescriptors.CalcNumRings(mol),
        "ArRings":  rdMolDescriptors.CalcNumAromaticRings(mol),
        "Ro5":      "PASS" if (Descriptors.ExactMolWt(mol)<=500 and
                               Descriptors.MolLogP(mol)<=5 and
                               rdMolDescriptors.CalcNumHBD(mol)<=5 and
                               rdMolDescriptors.CalcNumHBA(mol)<=10) else "FAIL"
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## Draw molecules as a grid

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import io

mols_draw = [Chem.MolFromSmiles(s) for s in library.values()]
legends   = list(library.keys())

img = Draw.MolsToGridImage(mols_draw, molsPerRow=4, subImgSize=(300, 200),
                            legends=legends, returnPNG=True)
with open("molecule_grid.png", "wb") as f:
    f.write(img)
display(Image("molecule_grid.png"))

## Substructure matching

In [ ]:
benzene   = Chem.MolFromSmarts("c1ccccc1")
carboxyl  = Chem.MolFromSmarts("[CX3](=O)[OX2H1]")
amine     = Chem.MolFromSmarts("[NX3;H1,H2]")

for name, smi in library.items():
    mol = Chem.MolFromSmiles(smi)
    has_benz  = mol.HasSubstructMatch(benzene)
    has_cooh  = mol.HasSubstructMatch(carboxyl)
    has_amine = mol.HasSubstructMatch(amine)
    print(f"{name:15s}: benzene={'Y' if has_benz else 'N'}  COOH={'Y' if has_cooh else 'N'}  NH={'Y' if has_amine else 'N'}")

## Key takeaways
- RDKit can load molecules from SMILES, InChI, SDF, and MOL formats
- Descriptors cover physicochemical, topological, and electronic properties
- SMARTS enables powerful substructure searching
- `Draw.MolsToGridImage` is the fastest way to visually inspect a compound library